# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and follows FAIR principles. You will use entity `@id`s throughout to reference record sets, fields, and columns.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Dataset Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get list of record sets using their @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Available record sets:")
    for record_set in record_sets:
        print(f"- @id: {record_set['@id']}, name: {record_set.get('name', '<no name>')}")

# For demo, list fields and columns for each record set if present
for record_set in record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    print("Fields:")
    for field in fields:
        print(f"  - {field['@id']} (name: {field.get('name', '')})")
        columns = field.get('column', [])
        if columns:
            print("    Columns:")
            for col in columns:
                print(f"      * {col['@id']} (name: {col.get('name', '')})")

# If record sets are empty, print a message
if not record_sets:
    print("No record sets available to preview.")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use `@id` for all references.

In [ ]:
# If there are record sets, extract data from the first one for demonstration
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nLoading records from record set @id: {first_record_set_id}")
    records = list(dataset.records(record_set=first_record_set_id))
    df = pd.DataFrame(records)
    print(f"Columns in DataFrame loaded from record set {first_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets found; cannot extract data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All references are made via `@id`.

In [ ]:
# Select a numeric field by its @id for demonstration
# For example, assume field @id 'log_likelihood' exists among columns
if record_sets and not df.empty:
    # Use a column which seems numeric; for demonstration, try columns that match
    # You may adjust 'log_likelihood' or use the actual @id from above overview
    numeric_field_id = None
    for col in df.columns:
        if 'log_likelihood' in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Fallback, try any numeric column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if numeric_field_id is not None:
        print(f"Using numeric field: {numeric_field_id}")
        # Set threshold as mean of field (arbitrary example)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_field_id = None
        # Look for a column with 'ward' or 'county' as possible grouping
        for col in df.columns:
            if 'ward' in col.lower() or 'county' in col.lower():
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found suitable for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib.

In [ ]:
# Visualize distribution of numeric field
if record_sets and not df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 6))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
This notebook explored the FAIR^2 dataset using Croissant schema and the `mlcroissant` API. We loaded the metadata, overviewed available record sets and fields by their `@id`, extracted record data, performed basic EDA and normalization, and visualized numeric distributions. 

This approach ensures consistent referencing, transparency, and reproducibility for FAIR datasets. You may further extend analysis with additional filtering or modeling using this foundation.